In [1]:
# Import Tools:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import seaborn as sns

plt.style.use('seaborn-v0_8-darkgrid')
pd.set_option('display.max_columns', None)

# IMPORT DATA
df_match = pd.read_csv(r'F:\ALMACENAMIENTO\DATA ANALITYCS\PORTAFOLIO\AI_Assistant_Hotels\SQL_ENRICHMENT\data_interaction__handling_hotel_match_features.csv', encoding='latin-1',dtype={'resolved': str})
df_full = pd.read_csv(r'F:\ALMACENAMIENTO\DATA ANALITYCS\PORTAFOLIO\AI_Assistant_Hotels\SQL_ENRICHMENT\data_interaction_hotel_full_features.csv',encoding='latin-1')


# Fix timestamp --> datetime
df_full['timestamp'] = pd.to_datetime(df_full['timestamp'], dayfirst=True)
df_match['timestamp'] = pd.to_datetime(df_match['timestamp'], dayfirst=True)

# Fix resolved en df_match
df_match['resolved'] = df_match['resolved'].map(
    {'True': True, 'False': False, 'TRUE': True, 'FALSE': False}
)

In [2]:

# ==========================================
# KPI CALCULATION - AI ASSISTANT IMPACT ANALYSIS
# ==========================================

import pandas as pd
import numpy as np

print("="*70)
print("KPI CALCULATION - AI ASSISTANT IMPACT ANALYSIS")
print("="*70)

# ==========================================
# A. PERFORMANCE METRICS
# ==========================================

print("\n" + "="*70)
print("A. PERFORMANCE METRICS (AI vs HUMAN)")
print("="*70)

# Separar datos por handler

df_ai_resolved = df_match[
    (df_match['handled_by'] == 'AI') & 
    (df_match['resolved'] == True)
]

df_h_resolved = df_match[
    (df_match['handled_by'] == 'Human') & 
    (df_match['resolved'] == True)
]

ai_data = df_match[df_match['handled_by'] == 'AI']
human_data = df_match[df_match['handled_by'] == 'Human']

# KPI 1: Average Response Time
ai_avg_response = df_ai_resolved['response_time_s'].mean()
human_avg_response = df_h_resolved['response_time_s'].mean()
overall_avg_response = df_match['response_time_s'].mean()

print(f"\n1. AVERAGE RESPONSE TIME:")
print(f"   AI:          {ai_avg_response:.1f} seconds ({ai_avg_response/60:.1f} minutes)")
print(f"   Human:       {human_avg_response:.1f} seconds ({human_avg_response/60:.1f} minutes)")
print(f"   Overall:     {overall_avg_response:.1f} seconds ({overall_avg_response/60:.1f} minutes)")

# KPI 2: Response Time Improvement
response_improvement = ((human_avg_response - ai_avg_response) / human_avg_response) * 100
time_saved_per_interaction = human_avg_response - ai_avg_response

print(f"\n2. RESPONSE TIME IMPROVEMENT (AI vs Human):")
print(f"   AI is {response_improvement:.1f}% faster than Human")
print(f"   Time saved per interaction: {time_saved_per_interaction:.1f} seconds ({time_saved_per_interaction/60:.1f} minutes)")

# KPI 3: Resolution Rate
ai_resolution = (ai_data['resolved'].sum() / len(ai_data)) * 100
human_resolution = (human_data['resolved'].sum() / len(human_data)) * 100
overall_resolution = (df_match['resolved'].sum() / len(df_match)) * 100

print(f"\n3. RESOLUTION RATE:")
print(f"   AI:          {ai_resolution:.1f}%")
print(f"   Human:       {human_resolution:.1f}%")
print(f"   Overall:     {overall_resolution:.1f}%")
print(f"   Improvement: {ai_resolution - human_resolution:.1f} percentage points")

# KPI 4: Fast Response Rate (<60s)
ai_fast = (df_ai_resolved['response_speed'] == 'Fast').sum() / len(df_ai_resolved) * 100
human_fast = (df_h_resolved['response_speed'] == 'Fast').sum() / len(df_h_resolved) * 100
overall_fast = (df_match['response_speed'] == 'Fast').sum() / len(df_match) * 100

print(f"\n4. FAST RESPONSE RATE (<60 seconds):")
print(f"   AI:          {ai_fast:.1f}%")
print(f"   Human:       {human_fast:.1f}%")
print(f"   Overall:     {overall_fast:.1f}%")

# ==========================================
# B. VOLUME METRICS
# ==========================================

print("\n" + "="*70)
print("B. VOLUME METRICS")
print("="*70)

# KPI 5: Total Interactions
total_interactions_full = len(df_full)
total_interactions_match = len(df_match)

print(f"\n5. TOTAL INTERACTIONS:")
print(f"   Total volume (df_full):      {total_interactions_full:,}")
print(f"   With handling data (df_match): {total_interactions_match:,}")
print(f"   Coverage:                    {total_interactions_match/total_interactions_full*100:.1f}%")

# KPI 6: AI Coverage Rate
ai_count = len(df_ai_resolved)
human_count = len(df_h_resolved)
ai_coverage = (ai_count / total_interactions_match) * 100
human_coverage = (human_count / total_interactions_match) * 100

print(f"\n6. HANDLER COVERAGE:")
print(f"   AI handled:      {ai_count:,} interactions ({ai_coverage:.1f}%)")
print(f"   Human handled:   {human_count:,} interactions ({human_coverage:.1f}%)")

# KPI 7: Channel Distribution
channel_dist = df_ai_resolved['channel'].value_counts()
print(f"\n7. CHANNEL DISTRIBUTION:")
for channel, count in channel_dist.items():
    pct = (count / len(df_ai_resolved)) * 100
    print(f"   {channel:15s}: {count:,} ({pct:.1f}%)")

# KPI 8: Peak Period Volume
peak_hours = df_match[df_match['hour_of_day'].isin([10, 11, 13])].shape[0]
peak_pct = (peak_hours / len(df_match)) * 100

print(f"\n8. PEAK PERIOD VOLUME (10-11 AM, 1 PM):")
print(f"   Peak hours:      {peak_hours:,} interactions ({peak_pct:.1f}%)")
print(f"   Off-peak hours:  {len(df_match) - peak_hours:,} interactions ({100-peak_pct:.1f}%)")

# ==========================================
# C. EFFICIENCY METRICS
# ==========================================

print("\n" + "="*70)
print("C. EFFICIENCY METRICS")
print("="*70)

# KPI 9: Total Time Saved by AI
total_time_saved = ai_count * time_saved_per_interaction
total_time_saved_hours = total_time_saved / 3600

print(f"\n9. TOTAL TIME SAVED BY AI:")
print(f"   Time saved: {total_time_saved:,.0f} seconds")
print(f"   Equivalent: {total_time_saved_hours:,.1f} hours")
print(f"   Equivalent: {total_time_saved_hours/8:.1f} working days (8h/day)")

# KPI 10: Potential Throughput Increase
ai_throughput = 3600 / ai_avg_response  # interactions per hour
human_throughput = 3600 / human_avg_response
throughput_increase = ((ai_throughput - human_throughput) / human_throughput) * 100

print(f"\n10. THROUGHPUT CAPACITY:")
print(f"   AI:          {ai_throughput:.1f} interactions/hour")
print(f"   Human:       {human_throughput:.1f} interactions/hour")
print(f"   AI advantage: {throughput_increase:.0f}% higher capacity")

# KPI 11: Workload Distribution by Complexity
complexity_dist = df_match.groupby(['handled_by', 'complexity']).size().unstack(fill_value=0)
print(f"\n11. WORKLOAD DISTRIBUTION BY COMPLEXITY:")
print(complexity_dist)

# ==========================================
# D. QUALITY METRICS
# ==========================================

print("\n" + "="*70)
print("D. QUALITY METRICS")
print("="*70)

# KPI 12: Service Level Achievement
sla_60s = (df_ai_resolved['response_time_s'] <= 60).sum() / len(df_ai_resolved) * 100
sla_300s = (df_ai_resolved['response_time_s'] <= 300).sum() / len(df_ai_resolved) * 100

print(f"\n12. SERVICE LEVEL ACHIEVEMENT:")
print(f"   Under 60s (Fast):     {sla_60s:.1f}%")
print(f"   Under 300s (Medium):  {sla_300s:.1f}%")

# KPI 13: Response Speed Distribution
speed_dist = df_ai_resolved['response_speed'].value_counts(normalize=True) * 100
print(f"\n13. RESPONSE SPEED DISTRIBUTION:")
for speed, pct in speed_dist.items():
    print(f"   {speed:10s}: {pct:.1f}%")

# KPI 14: Unresolved Cases
unresolved_ai = len(ai_data) - ai_data['resolved'].sum()
unresolved_human = len(human_data) - human_data['resolved'].sum()
unresolved_total = len(df_match) - df_match['resolved'].sum()

print(f"\n14. UNRESOLVED CASES:")
print(f"   AI:          {unresolved_ai} ({unresolved_ai/len(ai_data)*100:.1f}%)")
print(f"   Human:       {unresolved_human} ({unresolved_human/len(human_data)*100:.1f}%)")
print(f"   Total:       {unresolved_total} ({unresolved_total/len(df_match)*100:.1f}%)")

print("\n" + "="*70)
print("KPI CALCULATION COMPLETED")
print("="*70)

KPI CALCULATION - AI ASSISTANT IMPACT ANALYSIS

A. PERFORMANCE METRICS (AI vs HUMAN)

1. AVERAGE RESPONSE TIME:
   AI:          35.8 seconds (0.6 minutes)
   Human:       364.4 seconds (6.1 minutes)
   Overall:     175.2 seconds (2.9 minutes)

2. RESPONSE TIME IMPROVEMENT (AI vs Human):
   AI is 90.2% faster than Human
   Time saved per interaction: 328.5 seconds (5.5 minutes)

3. RESOLUTION RATE:
   AI:          98.8%
   Human:       72.9%
   Overall:     88.9%
   Improvement: 25.8 percentage points

4. FAST RESPONSE RATE (<60 seconds):
   AI:          80.4%
   Human:       0.1%
   Overall:     49.0%

B. VOLUME METRICS

5. TOTAL INTERACTIONS:
   Total volume (df_full):      4,740
   With handling data (df_match): 3,292
   Coverage:                    69.5%

6. HANDLER COVERAGE:
   AI handled:      2,004 interactions (60.9%)
   Human handled:   921 interactions (28.0%)

7. CHANNEL DISTRIBUTION:
   website_chat   : 831 (41.5%)
   email          : 681 (34.0%)
   facebook       : 270 (13.

In [5]:
# ==========================================
# E. ADDITIONAL KPIs
# ==========================================

print("\n" + "="*70)
print("E. ADDITIONAL KPIs")
print("="*70)

# KPI 15: Email interactions resolved by AI vs Human
ai_email = df_ai_resolved[df_ai_resolved['channel'] == 'email']
human_email = df_h_resolved[df_h_resolved['channel'] == 'email']

print(f"\n15. EMAIL RESOLVED COUNT:")
print(f"   AI resolved (email):     {len(ai_email):,}")
print(f"   Human resolved (email):  {len(human_email):,}")
print(f"   Total email resolved:    {len(ai_email) + len(human_email):,}")
print(f"   AI share of email:       {len(ai_email) / (len(ai_email) + len(human_email)) * 100:.1f}%")

# KPI 16: Speed replies — AI vs Human (X times faster)
speed_ratio = human_avg_response / ai_avg_response

print(f"\n16. SPEED REPLIES (AI vs Human):")
print(f"   AI avg response time:    {ai_avg_response:.1f}s")
print(f"   Human avg response time: {human_avg_response:.1f}s")
print(f"   AI is {speed_ratio:.1f}x faster than Human")

# KPI 17: Total cost saved based on time
hourly_rate = 350
cost_saved = total_time_saved_hours * hourly_rate

print(f"\n17. TOTAL COST SAVED:")
print(f"   Hours saved:             {total_time_saved_hours:,.1f} h")
print(f"   Hourly rate:             ${hourly_rate:,}/h")
print(f"   Total cost saved:        ${cost_saved:,.0f}")
print(f"   Equivalent (daily 8h):   ${hourly_rate * 8:,.0f}/day saved")


E. ADDITIONAL KPIs

15. EMAIL RESOLVED COUNT:
   AI resolved (email):     681
   Human resolved (email):  276
   Total email resolved:    957
   AI share of email:       71.2%

16. SPEED REPLIES (AI vs Human):
   AI avg response time:    35.8s
   Human avg response time: 364.4s
   AI is 10.2x faster than Human

17. TOTAL COST SAVED:
   Hours saved:             182.9 h
   Hourly rate:             $350/h
   Total cost saved:        $64,012
   Equivalent (daily 8h):   $2,800/day saved


In [4]:
# ==========================================
# EXPORT DATA TO EXCEL
# ==========================================

output_path = 'kpi_summary1.xlsx'

# ── Función para limpiar tipos Arrow ──
def clean_df(df):
    df = df.copy()
    for col in df.columns:
        try:
            df[col] = df[col].astype(str) if df[col].dtype == 'object' else df[col]
        except:
            df[col] = df[col].astype(str)
    return df.convert_dtypes(convert_string=False)

# ── KPI Summary ──
kpi_data = {
    'Category': [
        *['A. Performance'] * 12,
        *['B. Volume'] * 9,
        *['C. Efficiency'] * 6,
        *['D. Quality'] * 7,
    ],
    'KPI': [
        'Avg Response Time — AI (s)',
        'Avg Response Time — Human (s)',
        'Avg Response Time — Overall (s)',
        'Response Time Improvement (%)',
        'Time Saved per Interaction (s)',
        'Resolution Rate — AI (%)',
        'Resolution Rate — Human (%)',
        'Resolution Rate — Overall (%)',
        'Resolution Rate Improvement (pts)',
        'Fast Response Rate — AI (%)',
        'Fast Response Rate — Human (%)',
        'Fast Response Rate — Overall (%)',
        'Total Interactions — Full Dataset',
        'Total Interactions — Matched Dataset',
        'Coverage (%)',
        'AI Resolved Interactions',
        'Human Resolved Interactions',
        'AI Share of Resolved (%)',
        'Human Share of Resolved (%)',
        'Peak Hours Interactions',
        'Peak Hours Share (%)',
        'Total Time Saved (seconds)',
        'Total Time Saved (hours)',
        'Total Time Saved (working days)',
        'AI Throughput (interactions/hr)',
        'Human Throughput (interactions/hr)',
        'Throughput Advantage (%)',
        'SLA Under 60s — AI (%)',
        'SLA Under 300s — AI (%)',
        'Unresolved Cases — AI',
        'Unresolved Cases — Human',
        'Unresolved Cases — Total',
        'Unresolved Rate — AI (%)',
        'Unresolved Rate — Human (%)',
    ],
    'Value': [
        round(ai_avg_response, 1),
        round(human_avg_response, 1),
        round(overall_avg_response, 1),
        round(response_improvement, 1),
        round(time_saved_per_interaction, 1),
        round(ai_resolution, 1),
        round(human_resolution, 1),
        round(overall_resolution, 1),
        round(ai_resolution - human_resolution, 1),
        round(ai_fast, 1),
        round(human_fast, 1),
        round(overall_fast, 1),
        int(total_interactions_full),
        int(total_interactions_match),
        round(total_interactions_match / total_interactions_full * 100, 1),
        int(ai_count),
        int(human_count),
        round(ai_coverage, 1),
        round(human_coverage, 1),
        int(peak_hours),
        round(peak_pct, 1),
        round(total_time_saved, 0),
        round(total_time_saved_hours, 1),
        round(total_time_saved_hours / 8, 1),
        round(ai_throughput, 1),
        round(human_throughput, 1),
        round(throughput_increase, 0),
        round(sla_60s, 1),
        round(sla_300s, 1),
        int(unresolved_ai),
        int(unresolved_human),
        int(unresolved_total),
        round(unresolved_ai / len(ai_data) * 100, 1),
        round(unresolved_human / len(human_data) * 100, 1),
    ]
}

df_kpis = pd.DataFrame(kpi_data)

# ── Channel Distribution ──
channel_series = df_ai_resolved['channel'].value_counts()
channel_df = pd.DataFrame({
    'channel': channel_series.index.astype(str),
    'count':   channel_series.values.astype(int),
})
channel_df['pct'] = (channel_df['count'] / channel_df['count'].sum() * 100).round(1)

# ── Export ──
with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    df_kpis.to_excel(writer, sheet_name='KPI Summary', index=False)
    channel_df.to_excel(writer, sheet_name='Channel Distribution', index=False)
    clean_df(df_ai_resolved).to_excel(writer, sheet_name='AI Resolved', index=False)
    clean_df(df_h_resolved).to_excel(writer, sheet_name='Human Resolved', index=False)
    clean_df(df_match).to_excel(writer, sheet_name='df_match', index=False)

print(f"✅ Exported to: {output_path}")
print(f"   Sheets: KPI Summary | Channel Distribution | AI Resolved | Human Resolved | df_match")

✅ Exported to: kpi_summary1.xlsx
   Sheets: KPI Summary | Channel Distribution | AI Resolved | Human Resolved | df_match


In [6]:
# Ejecuta esto ANTES del export
ai_email = df_ai_resolved[df_ai_resolved['channel'] == 'email']
human_email = df_h_resolved[df_h_resolved['channel'] == 'email']
speed_ratio = human_avg_response / ai_avg_response
hourly_rate = 350
cost_saved = total_time_saved_hours * hourly_rate

In [7]:
# ── Export Additional KPIs to separate Excel ──
output_path_additional = 'kpi_additional.xlsx'

additional_kpi_data = {
    'Category': [*['E. Additional'] * 7],
    'KPI': [
        'Email Resolved — AI',
        'Email Resolved — Human',
        'Email Resolved — Total',
        'AI Share of Email (%)',
        'Speed Ratio — AI vs Human (x)',
        'Total Cost Saved (NOK)',
        'Hourly Rate (NOK)',
    ],
    'Value': [
        int(len(ai_email)),
        int(len(human_email)),
        int(len(ai_email) + len(human_email)),
        round(len(ai_email) / (len(ai_email) + len(human_email)) * 100, 1),
        round(speed_ratio, 1),
        round(cost_saved, 0),
        int(hourly_rate),
    ]
}

df_additional = pd.DataFrame(additional_kpi_data)

with pd.ExcelWriter(output_path_additional, engine='openpyxl') as writer:
    df_additional.to_excel(writer, sheet_name='Additional KPIs', index=False)

print(f"✅ Exported to: {output_path_additional}")

✅ Exported to: kpi_additional.xlsx
